# Lane pricing exploration

Exploratory work for the **rate cards** project (v1.1). Goal: per-lane base rates for LTL
that beat our current density-only pricing on margin without hurting win rate.

Inputs: accepted/declined quotes (Mar–Aug 2025), carrier cost per shipment, Pacific Fuel
Index weekly surcharge table (public CSV).

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA = Path("data")
quotes = pd.read_parquet(DATA / "quotes_2025-03_2025-08.parquet")
costs = pd.read_parquet(DATA / "carrier_costs_2025-03_2025-08.parquet")
fuel = pd.read_csv(DATA / "pacific_fuel_index_weekly.csv", parse_dates=["week_start"])
quotes["lane"] = quotes["origin_terminal"] + "→" + quotes["destination_terminal"]
quotes.shape, costs.shape, fuel.shape

((18422, 14), (6107, 6), (26, 3))

In [2]:
df = quotes.merge(costs, on="booking_reference", how="left")
df["margin_pct"] = (df["quoted_total"] - df["carrier_cost"]) / df["quoted_total"]
by_lane = (df[df["accepted"]]
           .groupby("lane")
           .agg(n=("lane", "size"), avg_cwt=("rate_per_cwt", "mean"), margin=("margin_pct", "median"))
           .query("n >= 30")
           .sort_values("margin"))
by_lane.head(10)

In [3]:
# Win-rate elasticity: logistic fit of accepted ~ price index per lane (rough!)
from sklearn.linear_model import LogisticRegression

def elasticity(g):
    if g["accepted"].nunique() < 2 or len(g) < 50:
        return np.nan
    x = (g["quoted_total"] / g["quoted_total"].median()).to_numpy().reshape(-1, 1)
    return LogisticRegression().fit(x, g["accepted"]).coef_[0][0]

lane_elasticity = df.groupby("lane").apply(elasticity).dropna().sort_values()
lane_elasticity.describe()

### Notes
* PCL→SEA and TAC→PDX are underpriced (median margin < 9%) with low elasticity → raise base ~6%.
* OAK→SLC is price-sensitive; keep current rates, revisit with fuel surcharge cap.
* Fuel surcharge should come from the live Pacific Fuel Index feed (`config/settings.yaml → fuel_index`)
  instead of the static 28.5% fallback.

TODO(samira): hand the lane table to Priya for `rate_cards` seed migration.